# Email Intent Classifier & Smart Reply Drafter

## 1. Project Overview

**Task:** Build a system that:
1. **Classifies** incoming emails by intent (complaint, inquiry, request, feedback, scheduling, urgent)
2. **Drafts** reply options in different tones (professional, friendly, concise)
3. **Structures** outputs as JSON for downstream integration

**Approach:** Prompt engineering with a local LLM — no fine-tuning needed. We design specialized prompts for classification, extraction, and generation tasks.

**Stack:**
- `LangChain` + `ChatOllama` — prompt templates and LLM interaction
- `Ollama` + `qwen3.5:9b` — local LLM (no cloud API keys)

> **No API keys required.** Everything runs locally via Ollama.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Design **classification prompts** that reliably categorize text into predefined labels |
| 2 | Extract **structured metadata** (intent, urgency, sentiment) from unstructured text |
| 3 | Use **tone-controlled generation** to produce style-varied outputs from the same input |
| 4 | Build **prompt templates** with placeholders for reusable pipelines |
| 5 | Get the LLM to output **valid JSON** — a critical skill for application integration |
| 6 | Evaluate classification accuracy on a labeled dataset |
| 7 | Understand when prompt engineering is sufficient vs when fine-tuning is needed |
| 8 | Chain multiple LLM calls into a coherent processing pipeline |

## 3. Problem Statement

A customer support team receives hundreds of emails daily. Each email must be:
- **Triaged** — is it a complaint? a question? a meeting request?
- **Prioritized** — how urgent is it?
- **Responded to** — with tone appropriate to the situation

Manual triage is slow and inconsistent across team members. This notebook automates intent classification and generates draft replies in multiple tones, letting the human pick or edit the best one.

## 4. Why This Project Matters

- **Email triage automation** is deployed at scale (Gmail Smart Reply, Zendesk auto-classification)
- **Tone control** is a key prompt engineering skill — the same content delivered in the wrong tone loses customers
- **Structured output** (getting LLMs to return JSON reliably) is essential for building real applications
- **Classification via prompting** is the fastest way to prototype intent systems before investing in fine-tuning

## 5. Pipeline Architecture

```
Incoming Email
      │
      ▼
  [Extract Metadata] ── intent, urgency, sentiment, key entities
      │
      ▼
  [Intent Classification] ── complaint / inquiry / request / feedback / scheduling / urgent
      │
      ▼
  [Draft Replies] ── generate 3 replies in different tones
      │
      ├──▶ Professional (formal business tone)
      ├──▶ Friendly (warm, approachable)
      └──▶ Concise (brief, to-the-point)
      │
      ▼
  [Structured Output] ── JSON with classification + replies
```

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core

print("Dependencies: langchain, langchain-ollama")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

In [ ]:
import os
import json
import re
import warnings
import time
from collections import Counter

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

print("All imports OK")

## 8. Configuration

In [ ]:
LLM_MODEL = "qwen3.5:9b"
LLM_TEMPERATURE = 0      # Deterministic for classification
GEN_TEMPERATURE = 0.5    # Slightly creative for reply drafting

# ── Intent categories ─────────────────────────────────
INTENT_LABELS = [
    "complaint",     # Customer is unhappy about a product or service
    "inquiry",       # Customer asking for information
    "request",       # Customer requesting an action (refund, change, etc.)
    "feedback",      # Customer sharing opinions or suggestions
    "scheduling",    # Meeting, appointment, or call scheduling
    "urgent",        # Time-sensitive issue needing immediate attention
]

# ── Tone options ──────────────────────────────────────
TONES = ["professional", "friendly", "concise"]

print("Configuration loaded")
print(f"  LLM: {LLM_MODEL}")
print(f"  Intents: {INTENT_LABELS}")
print(f"  Tones: {TONES}")

## 9. Sample Email Dataset

We create a diverse set of realistic emails with ground-truth labels. Each email is labeled with:
- **Intent** — the primary purpose of the email
- **Urgency** — low / medium / high
- **Sender** — context about who is writing

In production, these would come from an email inbox or CRM system.

In [ ]:
EMAILS = [
    # ── Complaints ────────────────────────────────────
    {
        "id": 1,
        "subject": "Terrible experience with your product",
        "body": "Hi,\n\nI purchased the ProMax 3000 blender two weeks ago and it has already broken down twice. The motor makes a grinding noise and the lid doesn't seal properly. I've been a loyal customer for 5 years but this is unacceptable. I want a full refund or a replacement immediately.\n\nVery disappointed,\nSarah M.",
        "sender": "sarah.m@email.com",
        "true_intent": "complaint",
        "true_urgency": "high",
    },
    {
        "id": 2,
        "subject": "Service outage affecting my business",
        "body": "Your cloud platform has been down for 6 hours now. Our entire team of 50 developers cannot deploy any code. We are losing revenue every minute. This is the third outage this quarter. We need an immediate resolution and a postmortem report. If this continues, we will be forced to evaluate alternatives.\n\nJames O., CTO",
        "sender": "james.o@techcorp.com",
        "true_intent": "complaint",
        "true_urgency": "high",
    },

    # ── Inquiries ─────────────────────────────────────
    {
        "id": 3,
        "subject": "Question about Enterprise pricing",
        "body": "Hello,\n\nWe're a 200-person company evaluating your Enterprise plan. Could you clarify:\n1. Is there a minimum contract length?\n2. Do you offer volume discounts for 200+ seats?\n3. Is SSO included or is it an add-on?\n\nWe'd like to schedule a call with your sales team if possible.\n\nBest regards,\nLisa K., VP of Engineering",
        "sender": "lisa.k@bigcorp.com",
        "true_intent": "inquiry",
        "true_urgency": "medium",
    },
    {
        "id": 4,
        "subject": "API rate limits",
        "body": "Hi team,\n\nI'm building an integration with your API and noticed the docs mention a rate limit of 100 requests/minute on the free tier. Does the paid plan have higher limits? Also, are there any batch endpoints for processing multiple items in a single call?\n\nThanks,\nDev Team at StartupXYZ",
        "sender": "dev@startupxyz.io",
        "true_intent": "inquiry",
        "true_urgency": "low",
    },

    # ── Requests ──────────────────────────────────────
    {
        "id": 5,
        "subject": "Requesting a refund for order #8834",
        "body": "Hello Support,\n\nI'd like to request a refund for order #8834, placed on March 15th. The item arrived damaged (cracked screen on the tablet). I've attached photos of the damage. Please process the refund to my original payment method.\n\nOrder details: Samsung Galaxy Tab A8, $279.99\n\nThank you,\nMike R.",
        "sender": "mike.r@gmail.com",
        "true_intent": "request",
        "true_urgency": "medium",
    },
    {
        "id": 6,
        "subject": "Please update my shipping address",
        "body": "Hi,\n\nI recently moved and need to update the shipping address on my account. My new address is:\n123 Maple Street, Apt 4B\nPortland, OR 97201\n\nPlease also update any pending orders that haven't shipped yet.\n\nThanks,\nAnna P.",
        "sender": "anna.p@outlook.com",
        "true_intent": "request",
        "true_urgency": "low",
    },

    # ── Feedback ──────────────────────────────────────
    {
        "id": 7,
        "subject": "Love the new dashboard update!",
        "body": "Hey team,\n\nJust wanted to say the new analytics dashboard is fantastic! The real-time charts are much faster and the export feature finally works the way I expected. One suggestion: it would be great if you could add a dark mode option. My team works late nights and the bright white background is harsh on the eyes.\n\nKeep up the great work!\nTom B.",
        "sender": "tom.b@company.co",
        "true_intent": "feedback",
        "true_urgency": "low",
    },
    {
        "id": 8,
        "subject": "Suggestions for your mobile app",
        "body": "Hi,\n\nI use your mobile app daily and have a few suggestions:\n- The search function is slow and doesn't support filters\n- Push notifications are too frequent and can't be customized\n- It would be nice to have offline mode for reading saved articles\n\nOverall the app is good but these improvements would make it much better.\n\nRegards,\nPreeti S.",
        "sender": "preeti.s@email.com",
        "true_intent": "feedback",
        "true_urgency": "low",
    },

    # ── Scheduling ────────────────────────────────────
    {
        "id": 9,
        "subject": "Demo call next week?",
        "body": "Hi Sales Team,\n\nWe discussed your platform at our last board meeting and there's strong interest. Could we schedule a 45-minute demo call next Tuesday or Wednesday? We'd have 3 people attending: myself, our CTO, and our Head of Product.\n\nWe're in EST timezone. Morning slots preferred.\n\nBest,\nRachel D., COO",
        "sender": "rachel.d@enterprise.com",
        "true_intent": "scheduling",
        "true_urgency": "medium",
    },
    {
        "id": 10,
        "subject": "Reschedule our onboarding session",
        "body": "Hello,\n\nUnfortunately I need to reschedule our onboarding session originally planned for Friday at 2pm. Something came up with a client. Would Monday at 10am or 3pm work instead?\n\nApologies for the inconvenience.\n\nCarlos V.",
        "sender": "carlos.v@startup.io",
        "true_intent": "scheduling",
        "true_urgency": "medium",
    },

    # ── Urgent ────────────────────────────────────────
    {
        "id": 11,
        "subject": "URGENT: Security breach detected",
        "body": "URGENT\n\nWe detected unauthorized access to our account at 3:42 AM today. Multiple API keys were generated that we did not authorize. We've changed our password but need immediate assistance to:\n1. Revoke all existing API keys\n2. Review access logs for the past 48 hours\n3. Enable additional security measures\n\nThis is a critical security incident requiring immediate attention.\n\nDr. Ahmed H., CISO",
        "sender": "ahmed.h@fintech.com",
        "true_intent": "urgent",
        "true_urgency": "high",
    },
    {
        "id": 12,
        "subject": "Data loss — need immediate backup recovery",
        "body": "Help!\n\nOur production database appears to have lost data from the last 24 hours after an update went wrong. We need to restore from backups immediately. This is affecting all our customers and our operations are at a standstill.\n\nPlease escalate to your highest priority.\n\nNaomi W., Head of Ops",
        "sender": "naomi.w@saascompany.com",
        "true_intent": "urgent",
        "true_urgency": "high",
    },
]

print(f"Dataset: {len(EMAILS)} emails")
intent_counts = Counter(e["true_intent"] for e in EMAILS)
for intent, count in intent_counts.most_common():
    print(f"  {intent}: {count}")

## 10. Data Inspection

Before building prompts, understand the data you're working with.

In [ ]:
# ── Show a few sample emails ──────────────────────────
for email in EMAILS[:3]:
    print(f"Email #{email['id']} | Intent: {email['true_intent']} | Urgency: {email['true_urgency']}")
    print(f"Subject: {email['subject']}")
    print(f"Body: {email['body'][:150]}...")
    print("-" * 60)

# ── Length statistics ─────────────────────────────────
body_lens = [len(e["body"]) for e in EMAILS]
print(f"\nEmail body length:")
print(f"  Min: {min(body_lens)} chars")
print(f"  Max: {max(body_lens)} chars")
print(f"  Mean: {sum(body_lens)/len(body_lens):.0f} chars")

## 11. Initialize the LLM

In [ ]:
llm = ChatOllama(model=LLM_MODEL, temperature=LLM_TEMPERATURE)
llm_creative = ChatOllama(model=LLM_MODEL, temperature=GEN_TEMPERATURE)

# Quick connectivity test
test = llm.invoke([HumanMessage(content="Respond with only: ready")])
raw = test.content
if "<think>" in raw:
    raw = raw.split("</think>")[-1].strip()
print(f"LLM ready: {raw[:50]}")

In [ ]:
def clean(text: str) -> str:
    """Strip thinking tags from model output."""
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str) -> dict | None:
    """Extract JSON from LLM output, handling markdown code blocks."""
    text = clean(text)
    # Strip markdown fences
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    # Find JSON object
    start = text.find("{")
    end = text.rfind("}") + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None

print("Helper functions defined")

## 12. Prompt Design — Intent Classification

### Why this prompt works

Good classification prompts have three key elements:

1. **Exhaustive label list with descriptions** — the LLM knows exactly what each category means
2. **Explicit constraint** — "return ONLY one of these labels" prevents creative answers
3. **No ambiguity** — the label set is mutually exclusive with clear boundaries

### Common mistake: vague labels

Bad: `["positive", "negative", "neutral"]` — too generic, no domain context.  
Good: `"complaint" — customer expressing dissatisfaction about a product or service issue` — specific and grounded.

In [ ]:
CLASSIFY_SYSTEM = """You are an expert email triage system for a customer support team.

Classify each email into EXACTLY ONE of these intents:
- "complaint" — customer expressing dissatisfaction about a product, service, or experience
- "inquiry" — customer asking for information, clarification, or details
- "request" — customer asking for a specific action (refund, update, change)
- "feedback" — customer sharing opinions, suggestions, or praise
- "scheduling" — arranging, changing, or cancelling meetings, calls, or appointments
- "urgent" — time-critical issue requiring immediate attention (security, data loss, outage)

Classification rules:
1. If an email contains both a complaint AND a request (e.g., "this is broken, I want a refund"), classify based on the PRIMARY purpose
2. "urgent" takes priority over other labels when there's a genuine emergency
3. Return ONLY the JSON object, no explanation"""

CLASSIFY_USER = """Classify this email.

Subject: {subject}
Body: {body}

Return JSON:
{{"intent": "<label>", "urgency": "low|medium|high", "confidence": "high|medium|low"}}"""

print("Classification prompt defined")
print(f"System prompt: {len(CLASSIFY_SYSTEM)} chars")

## 13. Run Intent Classification on All Emails

We classify every email and compare predictions against ground truth labels.

In [ ]:
classifications = []

print("Classifying emails...\n")
for email in EMAILS:
    resp = llm.invoke([
        SystemMessage(content=CLASSIFY_SYSTEM),
        HumanMessage(content=CLASSIFY_USER.format(
            subject=email["subject"], body=email["body"]
        ))
    ])

    parsed = parse_json(resp.content)
    if parsed is None:
        parsed = {"intent": "unknown", "urgency": "medium", "confidence": "low"}

    pred_intent = parsed.get("intent", "unknown").lower().strip()
    pred_urgency = parsed.get("urgency", "medium").lower().strip()
    confidence = parsed.get("confidence", "medium").lower().strip()

    match = "✅" if pred_intent == email["true_intent"] else "❌"
    print(f"  {match} #{email['id']:2d} | Pred: {pred_intent:12s} | True: {email['true_intent']:12s} | "
          f"Urgency: {pred_urgency:6s} | Conf: {confidence}")

    classifications.append({
        "id": email["id"],
        "true_intent": email["true_intent"],
        "pred_intent": pred_intent,
        "true_urgency": email["true_urgency"],
        "pred_urgency": pred_urgency,
        "confidence": confidence,
    })

print("\nClassification complete!")

## 14. Evaluation — Classification Accuracy

In [ ]:
# ── Intent accuracy ───────────────────────────────────
correct_intent = sum(1 for c in classifications if c["pred_intent"] == c["true_intent"])
total = len(classifications)
intent_acc = correct_intent / total

print(f"Intent Classification Accuracy: {correct_intent}/{total} = {intent_acc:.1%}")

# ── Urgency accuracy ──────────────────────────────────
correct_urgency = sum(1 for c in classifications if c["pred_urgency"] == c["true_urgency"])
urgency_acc = correct_urgency / total
print(f"Urgency Classification Accuracy: {correct_urgency}/{total} = {urgency_acc:.1%}")

# ── Per-intent breakdown ──────────────────────────────
print(f"\nPer-Intent Breakdown:")
for intent in INTENT_LABELS:
    true_count = sum(1 for c in classifications if c["true_intent"] == intent)
    correct = sum(1 for c in classifications
                  if c["true_intent"] == intent and c["pred_intent"] == intent)
    if true_count > 0:
        print(f"  {intent:12s}: {correct}/{true_count} correct")

# ── Misclassifications ────────────────────────────────
errors = [c for c in classifications if c["pred_intent"] != c["true_intent"]]
if errors:
    print(f"\nMisclassifications ({len(errors)}):")
    for e in errors:
        print(f"  Email #{e['id']}: predicted '{e['pred_intent']}' but true is '{e['true_intent']}'")
else:
    print("\nNo misclassifications — perfect accuracy!")

## 15. Prompt Design — Metadata Extraction

Beyond simple classification, we can extract **structured metadata** from emails: key entities, action items, sentiment, and a one-line summary.

### Why use a JSON schema in the prompt?

Showing the LLM the **exact JSON structure** you want (with field names as hints) is far more reliable than describing it in prose. The schema acts as a template that the LLM fills in.

In [ ]:
EXTRACT_SYSTEM = """You are an email analysis engine that extracts structured metadata.
Return ONLY valid JSON. No explanation, no markdown outside the JSON."""

EXTRACT_USER = """Analyze this email and extract metadata.

Subject: {subject}
Body: {body}

Return this exact JSON structure:
{{
  "summary": "one-sentence summary of the email",
  "intent": "complaint|inquiry|request|feedback|scheduling|urgent",
  "sentiment": "positive|negative|neutral|mixed",
  "urgency": "low|medium|high",
  "key_entities": ["product names, order numbers, dates, names mentioned"],
  "action_required": "what specific action the sender wants",
  "suggested_department": "support|sales|engineering|security|billing"
}}"""

print("Metadata extraction prompt defined")

In [ ]:
# ── Extract metadata for a sample of emails ───────────
sample_emails = [EMAILS[0], EMAILS[2], EMAILS[10]]  # complaint, inquiry, urgent

for email in sample_emails:
    print(f"Email #{email['id']}: {email['subject']}")
    print("-" * 50)

    resp = llm.invoke([
        SystemMessage(content=EXTRACT_SYSTEM),
        HumanMessage(content=EXTRACT_USER.format(
            subject=email["subject"], body=email["body"]
        ))
    ])

    parsed = parse_json(resp.content)
    if parsed:
        print(json.dumps(parsed, indent=2))
    else:
        print(f"  (Could not parse JSON from: {clean(resp.content)[:200]})")
    print()

## 16. Prompt Design — Tone-Controlled Reply Drafting

### The key insight about tone control

Tone isn't just about word choice — it affects:
- **Sentence structure** — formal uses complex sentences; concise uses short ones
- **Greetings/closings** — "Dear Sarah" vs "Hi Sarah!" vs just "Hi,"
- **Hedging language** — "We sincerely apologize" vs "Sorry about that!" vs "Apologies."
- **Paragraph length** — professional is paragraphed; concise is bullets

The prompt must describe each tone with enough detail that the LLM can consistently reproduce it.

### Tone definitions

| Tone | Description | Best For |
|------|-------------|----------|
| **Professional** | Formal, corporate language. Full sentences, proper greetings/closings. | Enterprise clients, legal, executive |
| **Friendly** | Warm, empathetic, conversational. Shows personality and care. | Consumer products, community, B2C |
| **Concise** | Brief, direct, no filler. Gets to the point in 2-3 sentences. | Internal comms, simple confirmations |

In [ ]:
TONE_DESCRIPTIONS = {
    "professional": (
        "Write in a formal, corporate tone. Use full sentences, proper greetings "
        "(Dear [Name]), and professional closings (Best regards). Be respectful, "
        "measured, and thorough. Acknowledge concerns diplomatically. Use "
        "phrases like 'We appreciate your patience' and 'Please do not hesitate "
        "to contact us'."
    ),
    "friendly": (
        "Write in a warm, approachable tone. Use casual greetings (Hi [Name]!), "
        "show empathy and enthusiasm. Be personable and supportive. Use "
        "contractions (we're, you'll). Include phrases like 'I totally understand' "
        "and 'We've got you covered!'. End warmly."
    ),
    "concise": (
        "Write as briefly as possible. Maximum 3-4 sentences. No filler words, "
        "no pleasantries beyond a short greeting. State the key information and "
        "action items directly. Use bullet points if multiple items. "
        "Example: 'Hi [Name], Confirmed. Your refund is processing (3-5 days). "
        "Questions? Reply here.'"
    ),
}

REPLY_SYSTEM = """You are a customer support agent drafting email replies.
Rules:
- Address the sender's specific concerns
- Use the sender's name if available
- Never promise what you can't deliver
- Include a clear next step or call to action
- Do NOT include a subject line — just the body
- Match the specified tone exactly"""

REPLY_USER = """Draft a reply to this email.

TONE: {tone_description}

Original email:
Subject: {subject}
From: {sender}
Body: {body}

Email intent: {intent}

Draft reply:"""

print("Reply drafting prompt defined")
print(f"Tone descriptions: {list(TONE_DESCRIPTIONS.keys())}")

## 17. Generate Reply Drafts — All Three Tones

We pick a few representative emails and generate replies in each tone. This demonstrates how the same content changes dramatically with different tone instructions.

In [ ]:
def draft_replies(email: dict, intent: str) -> dict:
    """Generate replies in all three tones for a given email."""
    replies = {}
    for tone_name, tone_desc in TONE_DESCRIPTIONS.items():
        resp = llm_creative.invoke([
            SystemMessage(content=REPLY_SYSTEM),
            HumanMessage(content=REPLY_USER.format(
                tone_description=tone_desc,
                subject=email["subject"],
                sender=email["sender"],
                body=email["body"],
                intent=intent,
            ))
        ])
        replies[tone_name] = clean(resp.content)
    return replies

print("draft_replies() function defined")

In [ ]:
# ── Generate replies for a complaint email ────────────
demo_email = EMAILS[0]  # Sarah's blender complaint
print(f"ORIGINAL EMAIL (#{demo_email['id']}):")
print(f"Subject: {demo_email['subject']}")
print(f"From: {demo_email['sender']}")
print(f"{demo_email['body']}")
print()

replies = draft_replies(demo_email, "complaint")

for tone_name, reply in replies.items():
    print(f"\n{'=' * 60}")
    print(f"REPLY — {tone_name.upper()} TONE")
    print("=" * 60)
    print(reply)

In [ ]:
# ── Generate replies for an inquiry email ─────────────
demo_email2 = EMAILS[2]  # Lisa's enterprise pricing inquiry
print(f"ORIGINAL EMAIL (#{demo_email2['id']}):")
print(f"Subject: {demo_email2['subject']}")
print(f"From: {demo_email2['sender']}")
print(f"{demo_email2['body'][:200]}...")
print()

replies2 = draft_replies(demo_email2, "inquiry")

for tone_name, reply in replies2.items():
    print(f"\n{'=' * 60}")
    print(f"REPLY — {tone_name.upper()} TONE")
    print("=" * 60)
    print(reply)

In [ ]:
# ── Generate replies for an urgent email ───────────────
demo_email3 = EMAILS[10]  # Security breach
print(f"ORIGINAL EMAIL (#{demo_email3['id']}):")
print(f"Subject: {demo_email3['subject']}")
print(f"From: {demo_email3['sender']}")
print(f"{demo_email3['body'][:200]}...")
print()

replies3 = draft_replies(demo_email3, "urgent")

for tone_name, reply in replies3.items():
    print(f"\n{'=' * 60}")
    print(f"REPLY — {tone_name.upper()} TONE")
    print("=" * 60)
    print(reply)

## 18. Full Pipeline — End-to-End Processing

Combine classification + extraction + reply drafting into one reusable function.

In [ ]:
def process_email(email: dict, verbose: bool = True) -> dict:
    """
    Full pipeline: classify → extract metadata → draft replies in 3 tones.
    """
    t0 = time.perf_counter()

    # Step 1: Classify intent
    resp = llm.invoke([
        SystemMessage(content=CLASSIFY_SYSTEM),
        HumanMessage(content=CLASSIFY_USER.format(
            subject=email["subject"], body=email["body"]
        ))
    ])
    cls = parse_json(resp.content) or {"intent": "unknown", "urgency": "medium"}
    intent = cls.get("intent", "unknown")

    # Step 2: Extract metadata
    resp2 = llm.invoke([
        SystemMessage(content=EXTRACT_SYSTEM),
        HumanMessage(content=EXTRACT_USER.format(
            subject=email["subject"], body=email["body"]
        ))
    ])
    metadata = parse_json(resp2.content) or {}

    # Step 3: Draft replies in all tones
    replies = draft_replies(email, intent)

    elapsed = time.perf_counter() - t0

    result = {
        "email_id": email["id"],
        "classification": cls,
        "metadata": metadata,
        "replies": replies,
        "processing_time_sec": round(elapsed, 1),
    }

    if verbose:
        print(f"Email #{email['id']}: {email['subject']}")
        print(f"  Intent: {intent} | Urgency: {cls.get('urgency', '?')}")
        if metadata:
            print(f"  Summary: {metadata.get('summary', 'N/A')}")
            print(f"  Sentiment: {metadata.get('sentiment', 'N/A')}")
            print(f"  Department: {metadata.get('suggested_department', 'N/A')}")
        print(f"  Replies generated: {list(replies.keys())}")
        print(f"  Time: {elapsed:.1f}s")

    return result

print("process_email() pipeline function defined")

In [ ]:
# ── Process all emails through the full pipeline ──────
print("Processing all emails through the full pipeline...\n")

all_results = []
for email in EMAILS:
    result = process_email(email)
    all_results.append(result)
    print()

## 19. Pipeline Dashboard

In [ ]:
print("EMAIL PROCESSING DASHBOARD")
print("=" * 60)
print(f"Total emails processed: {len(all_results)}")

# Intent distribution
pred_intents = Counter(r["classification"].get("intent", "unknown") for r in all_results)
print(f"\nPredicted intent distribution:")
for intent, count in pred_intents.most_common():
    print(f"  {intent:12s}: {count}")

# Urgency distribution
urgencies = Counter(r["classification"].get("urgency", "unknown") for r in all_results)
print(f"\nUrgency distribution:")
for urg, count in urgencies.most_common():
    print(f"  {urg:8s}: {'🔴' if urg == 'high' else '🟡' if urg == 'medium' else '🟢'} {count}")

# Timing
times = [r["processing_time_sec"] for r in all_results]
print(f"\nProcessing time:")
print(f"  Total: {sum(times):.0f}s")
print(f"  Avg per email: {sum(times)/len(times):.1f}s")
print(f"  Min: {min(times):.1f}s | Max: {max(times):.1f}s")

# Departments
depts = Counter(r["metadata"].get("suggested_department", "unknown") for r in all_results)
print(f"\nSuggested department routing:")
for dept, count in depts.most_common():
    print(f"  {dept:12s}: {count}")

## 20. Tone Analysis — Comparing Reply Styles

Let's quantitatively compare how tone affects reply characteristics.

In [ ]:
print("TONE COMPARISON ANALYSIS")
print("=" * 60)

for tone in TONES:
    lengths = [len(r["replies"][tone]) for r in all_results if tone in r["replies"]]
    word_counts = [len(r["replies"][tone].split()) for r in all_results if tone in r["replies"]]
    print(f"\n{tone.upper()} tone:")
    print(f"  Avg length: {sum(lengths)/len(lengths):.0f} chars")
    print(f"  Avg words:  {sum(word_counts)/len(word_counts):.0f}")
    print(f"  Min words:  {min(word_counts)}")
    print(f"  Max words:  {max(word_counts)}")

# ── Side-by-side length comparison for each email ─────
print(f"\nPer-Email Word Counts (Professional / Friendly / Concise):")
for r in all_results:
    eid = r["email_id"]
    counts = [len(r["replies"].get(t, "").split()) for t in TONES]
    bar = " | ".join(f"{c:3d}" for c in counts)
    print(f"  #{eid:2d}: {bar}")

## 21. Error Analysis & Edge Cases

In [ ]:
# ── Edge case 1: Ambiguous email ──────────────────────
ambiguous_email = {
    "id": 99,
    "subject": "Re: Account",
    "body": "Following up on my previous message. Things still aren't working. Also, when are we meeting?",
    "sender": "unclear@example.com",
    "true_intent": "complaint",
    "true_urgency": "medium",
}

print("EDGE CASE 1: Ambiguous email (complaint + scheduling)")
print("-" * 50)
result_ambiguous = process_email(ambiguous_email)

# ── Edge case 2: Very short email ─────────────────────
short_email = {
    "id": 98,
    "subject": "Refund",
    "body": "Where is my refund?",
    "sender": "terse@example.com",
    "true_intent": "request",
    "true_urgency": "medium",
}

print("\nEDGE CASE 2: Very short email")
print("-" * 50)
result_short = process_email(short_email)

# ── Edge case 3: Positive complaint (mixed signals) ───
mixed_email = {
    "id": 97,
    "subject": "Mostly great but one issue",
    "body": "I absolutely love your product and recommend it to everyone. However, the latest update broke the export feature that my team relies on daily. Can you prioritize fixing this? Everything else is perfect.",
    "sender": "fan@example.com",
    "true_intent": "feedback",
    "true_urgency": "medium",
}

print("\nEDGE CASE 3: Positive email with embedded complaint")
print("-" * 50)
result_mixed = process_email(mixed_email)

### Limitations

| Limitation | Explanation | Mitigation |
|------------|-------------|------------|
| **Ambiguous intents** | Some emails genuinely span multiple categories | Allow multi-label classification or "primary + secondary" intent |
| **Context-dependent tone** | The "right" tone depends on company culture and sender relationship | Make tone selection configurable per-customer or per-department |
| **Hallucinated details** | Replies may include promises not in company policy | Add a policy document to the system prompt; human review before sending |
| **No conversation context** | Each email is processed independently — no thread context | Include previous email thread in the prompt |
| **Sarcasm / irony** | "Great job breaking my account" is sarcasm, not praise | Hard to solve; LLMs handle obvious sarcasm but miss subtle cases |
| **Non-English emails** | Prompts are English-only | Add language detection and translate before processing |

## 22. Common Mistakes

| Mistake | Why It's Wrong | What to Do Instead |
|---------|---------------|-------------------|
| **Not describing label definitions** | LLM interprets labels differently without context | Write 1-sentence definition for each intent label |
| **Mixing classification and generation** | One prompt for "classify and reply" degrades both tasks | Use separate prompts: classify first, then draft |
| **Vague tone instructions** | "Be nice" produces inconsistent results | Describe tone with specific examples of greetings, word choice, structure |
| **No output format specification** | LLM returns prose instead of JSON, or JSON wrapped in explanation | Show the exact JSON schema in the prompt |
| **Using high temperature for classification** | Adds randomness to what should be a deterministic decision | Use temperature=0 for classification; 0.3-0.7 for generation |
| **No handling of parse failures** | JSON parsing crashes the pipeline on malformed output | Always wrap parse in try/except with sensible defaults |
| **Sending AI-drafted replies without review** | Hallucinated promises, wrong tone, factual errors reach customers | Always have a human review step before sending |
| **Not testing edge cases** | Pipeline works on clean examples but fails on short/ambiguous/mixed emails | Test with adversarial and boundary examples |

## 23. How to Improve This Project

| Improvement | Difficulty | Expected Impact |
|-------------|------------|----------------|
| **Few-shot classification** (add examples per intent) | Easy | Higher accuracy on ambiguous emails |
| **Thread context** (include previous emails) | Medium | Better replies that reference conversation history |
| **Custom tone profiles** (per customer or brand) | Easy | More appropriate tone matching |
| **Policy grounding** (inject company policies into system prompt) | Easy | Replies that follow actual company rules |
| **Confidence thresholding** (flag low-confidence for human review) | Easy | Reduce errors on hard cases |
| **Multi-label classification** (allow multiple intents) | Medium | Handle emails that span categories |
| **Fine-tuned classifier** (train on labeled email dataset) | Hard | Faster, cheaper, and more accurate at scale |
| **A/B testing** (compare generated replies vs human-written) | Medium | Measure real-world effectiveness |

## 24. Production Improvement Ideas

1. **Email integration** — Connect to Gmail/Outlook API to process incoming emails automatically
2. **Reply approval workflow** — Route drafts through a queue where agents pick, edit, and send
3. **Template library** — Maintain approved reply templates per intent; LLM personalizes them
4. **Analytics dashboard** — Track intent distribution, response times, and tone preferences
5. **Feedback loop** — Agents rate AI drafts → data feeds back to improve prompts
6. **Escalation rules** — Auto-route "urgent" + "high confidence" to on-call staff
7. **CRM integration** — Pull customer history to personalize replies
8. **Batch processing** — Process email queue during off-hours, drafts ready by morning

## 25. Exercises — Try These Yourself

### Exercise 1: Add Few-Shot Examples

Add 1 example per intent to the classification prompt and test whether accuracy improves.

In [ ]:
# ── Exercise 1: Few-shot classification ───────────────
FEW_SHOT_CLASSIFY = """Classify this email. Here are examples:

Example 1:
Subject: "Product is broken"
Body: "My device stopped working after 2 days. Not happy."
→ {{"intent": "complaint", "urgency": "medium", "confidence": "high"}}

Example 2:
Subject: "How does pricing work?"
Body: "Can you explain your team pricing for 50 users?"
→ {{"intent": "inquiry", "urgency": "low", "confidence": "high"}}

Example 3:
Subject: "URGENT: System down"
Body: "Production server is unresponsive. All customers affected."
→ {{"intent": "urgent", "urgency": "high", "confidence": "high"}}

Now classify:
Subject: {subject}
Body: {body}

Return JSON: {{"intent": "<label>", "urgency": "low|medium|high", "confidence": "high|medium|low"}}"""

# Test on a tricky email
test_email = EMAILS[4]  # refund request
resp = llm.invoke([
    SystemMessage(content=CLASSIFY_SYSTEM),
    HumanMessage(content=FEW_SHOT_CLASSIFY.format(
        subject=test_email["subject"], body=test_email["body"]
    ))
])
parsed = parse_json(resp.content)
print(f"Few-shot result: {parsed}")
print(f"True intent: {test_email['true_intent']}")
print("\nTry adding examples for 'request', 'feedback', and 'scheduling' intents too!")

### Exercise 2: Add a New Tone

Create an "empathetic" tone for handling complaint emails, and compare it against the existing tones.

In [ ]:
# ── Exercise 2: Custom tone ───────────────────────────
EMPATHETIC_TONE = (
    "Write with deep empathy and understanding. Acknowledge the customer's "
    "frustration explicitly. Use phrases like 'I completely understand how "
    "frustrating this must be' and 'You have every right to be upset'. "
    "Show that you take their issue personally. Offer a clear resolution "
    "and follow-up timeline. Be human, not corporate."
)

# Generate empathetic reply for the complaint
complaint = EMAILS[0]
resp = llm_creative.invoke([
    SystemMessage(content=REPLY_SYSTEM),
    HumanMessage(content=REPLY_USER.format(
        tone_description=EMPATHETIC_TONE,
        subject=complaint["subject"],
        sender=complaint["sender"],
        body=complaint["body"],
        intent="complaint",
    ))
])

print("EMPATHETIC TONE REPLY:")
print("=" * 60)
print(clean(resp.content))
print("\nCompare this with the professional and friendly replies above.")
print("Notice how empathy differs from friendliness — it validates feelings, not just tone.")

### Exercise 3: Structured JSON Output for All Pipeline Steps

In [ ]:
# ── Exercise 3: Full structured output ────────────────
# Try getting the LLM to produce the ENTIRE pipeline output as one JSON call
# This is harder but tests your prompt engineering skills

ALL_IN_ONE_PROMPT = """Analyze this email and return a COMPLETE JSON response.

Subject: {subject}
Body: {body}

Return this EXACT JSON structure (no other text):
{{
  "classification": {{
    "intent": "complaint|inquiry|request|feedback|scheduling|urgent",
    "urgency": "low|medium|high",
    "sentiment": "positive|negative|neutral|mixed"
  }},
  "summary": "one-sentence summary",
  "suggested_reply": "a brief professional reply (3-5 sentences)"
}}"""

test = EMAILS[6]  # positive feedback
resp = llm.invoke([
    SystemMessage(content="Return ONLY valid JSON. No explanation."),
    HumanMessage(content=ALL_IN_ONE_PROMPT.format(
        subject=test["subject"], body=test["body"]
    ))
])

parsed = parse_json(resp.content)
if parsed:
    print("All-in-one JSON output:")
    print(json.dumps(parsed, indent=2))
else:
    print(f"Failed to parse. Raw: {clean(resp.content)[:300]}")
    print("\nTip: Complex JSON is harder for LLMs. Separate calls are more reliable.")

### Mini Challenge

1. **Write your own emails:** Add 3 new emails to the dataset with different intents. Run the pipeline. Does the classifier handle them correctly?

2. **Multi-language support:** Modify the pipeline to detect email language first, then translate to English before classification. How would you structure this as a prompt chain?

3. **Reply ranking:** After generating 3 toned replies, ask the LLM to rank them for appropriateness given the email's intent and urgency. Which tone does it prefer for complaints vs inquiries?

4. **Confidence calibration:** Send the same email 5 times and check if the classification is consistent. Does the LLM give different answers with temperature=0? (It shouldn't — but sometimes does.)

## 26. Session Summary

In [ ]:
print("SESSION SUMMARY")
print("=" * 50)

# Classification accuracy
correct = sum(1 for c in classifications if c["pred_intent"] == c["true_intent"])
print(f"Intent accuracy:   {correct}/{len(classifications)} ({correct/len(classifications):.0%})")

correct_u = sum(1 for c in classifications if c["pred_urgency"] == c["true_urgency"])
print(f"Urgency accuracy:  {correct_u}/{len(classifications)} ({correct_u/len(classifications):.0%})")

# Total emails processed
print(f"\nEmails in dataset: {len(EMAILS)}")
print(f"Full pipeline runs: {len(all_results)}")
print(f"Replies generated: {len(all_results) * 3} (3 tones × {len(all_results)} emails)")
print(f"Edge cases tested: 3")

# LLM details
print(f"\nLLM: {LLM_MODEL}")
print(f"Classification temp: {LLM_TEMPERATURE}")
print(f"Generation temp: {GEN_TEMPERATURE}")
total_time = sum(r["processing_time_sec"] for r in all_results)
print(f"Total processing time: {total_time:.0f}s")

## 27. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Classification prompts need label definitions** — not just label names, but 1-sentence descriptions of what each means |
| 2 | **Separate classification from generation** — one prompt per task is more reliable than one mega-prompt |
| 3 | **Tone control requires specific instructions** — "professional" is vague; describe greetings, word choice, and structure explicitly |
| 4 | **JSON schemas in prompts** produce structured output far more reliably than verbal descriptions |
| 5 | **Temperature matters** — use 0 for classification (deterministic), 0.3–0.7 for generation (creative) |
| 6 | **Always handle parse failures gracefully** — LLMs occasionally produce malformed JSON |
| 7 | **Edge cases reveal prompt weaknesses** — ambiguous, short, and mixed-signal emails expose gaps |
| 8 | **Human review is non-negotiable** — AI drafts are suggestions, not final replies |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*